In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna

print("라이브러리 불러오기")

라이브러리 불러오기


In [2]:
#windows 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [3]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)

In [4]:
df["target_original"] = data.target

In [5]:
df["target"] = 1 - df["target_original"]

In [6]:
df["target_name"] = df["target"].map({
    0: "benign",
    1: "malignant"
})

In [7]:
print("데이터 크기 (행, 열):", df.shape)
print("\n원본 target 기준 개수:")
print(df["target_original"].value_counts().sort_index())
print("\n수업용 target 기준 개수:")
print(df["target"].value_counts().sort_index())
print("\n수업용 target 이름 기준 개수:")
print(df["target_name"].value_counts())

데이터 크기 (행, 열): (569, 33)

원본 target 기준 개수:
target_original
0    212
1    357
Name: count, dtype: int64

수업용 target 기준 개수:
target
0    357
1    212
Name: count, dtype: int64

수업용 target 이름 기준 개수:
target_name
benign       357
malignant    212
Name: count, dtype: int64


In [8]:
# 모델 입력값 X를 만듭니다
X = df.drop(columns=["target_original", "target", "target_name"])

# 모델이 맞혀야 하는 정답 y를 만듭니다.
y = df["target"]

print("X 크기:", X.shape)
print("y 크기:", y.shape)

X 크기: (569, 30)
y 크기: (569,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train 크기:", X_train.shape)
print("test 크기:", X_test.shape)
print("\ntrain의 target 비율:")
print(y_train.value_counts(normalize=True).round(3))
print("\ntest의 target 비율:")
print(y_test.value_counts(normalize=True).round(3))

train 크기: (455, 30)
test 크기: (114, 30)

train의 target 비율:
target
0    0.626
1    0.374
Name: proportion, dtype: float64

test의 target 비율:
target
0    0.632
1    0.368
Name: proportion, dtype: float64


In [10]:
def evaluate_classification_model(model_name, y_true, y_pred, y_proba):
    """
    분류 모델의 성능을 같은 기준으로 평가하는 함수입니다.

    이 수업에서는 target을 다음 기준으로 사용합니다.

    0 = benign, 양성
    1 = malignant, 악성

    따라서 precision, recall, f1-score는
    malignant, 즉 1번 클래스를 기준으로 계산합니다.

    model_name : 모델 이름 (비교표에서 구분용)
    y_true     : 실제 정답
    y_pred     : 모델이 예측한 0/1 값
    y_proba    : malignant(=1)일 확률 (predict_proba(X)[:, 1])
    """
    # 전체 중 맞춘 비율 (단, accuracy 하나만 보면 안 됩니다)
    accuracy = accuracy_score(y_true, y_pred)

    # malignant(=1)로 예측한 것 중 실제로 malignant인 비율
    precision_malignant = precision_score(y_true, y_pred, pos_label=1)

    # 실제 malignant 중 모델이 malignant로 맞춘 비율 (놓치지 않는 능력)
    recall_malignant = recall_score(y_true, y_pred, pos_label=1)

    # precision과 recall의 균형 점수
    f1_malignant = f1_score(y_true, y_pred, pos_label=1)

    # 확률 기준 전반적인 구분 능력 (threshold 하나에 묶이지 않음)
    roc_auc = roc_auc_score(y_true, y_proba)

    # [[TN, FP]
    #  [FN, TP]]

    # TN : 실제 benign이고 benign으로 예측
    # FP : 실제 benign인데 amlignant로 예측한 경우
    # FN : 실제 malignant인데 benign으로 예측한 경우 <- 의료에서 특히 위험***
    # TP : 실제 malignant이고 malignant로 예측한 경우
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    result = {
        "model_name": model_name,
        "accuracy": accuracy,
        "precision_malignant": precision_malignant,
        "recall_malignant": recall_malignant,
        "f1_malignant": f1_malignant,
        "roc_auc": roc_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    return result

print("평가 함수 준비 완료")

평가 함수 준비 완료


In [11]:
xgb_base_model = XGBClassifier(
    n_estimators=100,       #트리 100개
    max_depth=3,            #트리 깊이 3
    learning_rate=0.1,      #학습 속도(보폭)
    random_state=42,        #재현성 고정
    eval_metric="logloss"   #학습 중 사용할 평가 기준
)

In [12]:
# train 데이터로 학습
xgb_base_model.fit(X_train, y_train)
# test 데이터로 예측
xgb_base_pred = xgb_base_model.predict(X_test)
# predict_proba()는 0/1로 정하기 전의 확률을 돌려줌
xgb_base_proba = xgb_base_model.predict_proba(X_test)[:, 1]

In [13]:
# 공통 평가 함수로 성능을 정리
xgb_base_result = evaluate_classification_model(
    model_name="XGBoost Base",
    y_true=y_test,
    y_pred=xgb_base_pred,
    y_proba=xgb_base_proba
)

xgb_base_result

{'model_name': 'XGBoost Base',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9966931216931217,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [14]:
# 더 자세한 평가 결과
print("=== Classification Report (XGBoost Base) ===")
print(classification_report(
    y_test,
    xgb_base_pred,
    target_names=["benign(0)", "malignant(1)"]
))

print("=== Confusion Matrix (XGBoost Base) ===")
print(confusion_matrix(y_test, xgb_base_pred))
print("\n[[TN, FP],]")
print(" [FN, TP]] 순서입니다.")

=== Classification Report (XGBoost Base) ===
              precision    recall  f1-score   support

   benign(0)       0.95      1.00      0.97        72
malignant(1)       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

=== Confusion Matrix (XGBoost Base) ===


[[72  0]
 [ 4 38]]

[[TN, FP],]
 [FN, TP]] 순서입니다.


In [15]:
# 튜닝에 사용할 XGBoost (설정값은 비워두고 탐색으로 채울 예정)
xgb_for_random_search = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

In [16]:
# 탐색할 하이퍼파라미터 후보 (수업시간을 고려해 너무 넓지 않게 제한)(*했지만 알아서 알잘딱으로 정하기*)
"""
param_distributions = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1.0],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0]
}
"""
param_distributions = {
    # --- 기존 유지 및 확장 ---
    "n_estimators": [50, 100, 150, 200, 300, 500],       # 대형 데이터셋을 위해 300, 500 추가 추천
    "max_depth": [2, 3, 4, 5, 6, 8],                     # 복잡한 패턴 학습을 위해 6, 8 추가 추천
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],       # 조금 더 빠른 학습 속도를 위해 0.2 추가 추천
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],              # 무작위성 확장을 위해 0.6 추가 추천
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],       # 0.6 추가 추천
    "reg_alpha": [0, 0.01, 0.1, 1.0, 10.0],              # 강한 규제를 위해 10.0 추가 추천
    "reg_lambda": [0.5, 1.0, 1.5, 2.0, 5.0, 10.0],       # 강한 규제를 위해 5.0, 10.0 추가 추천
    
    # --- 새로 추가하면 좋은 핵심 파라미터 ---
    "min_child_weight": [1, 3, 5, 7],                    # 과적합 방지 끝판왕 (값이 클수록 모델이 보수적으로 변함)
    "gamma": [0, 0.1, 0.2, 0.4, 0.8],                    # 리프 노드를 추가로 분할할지 결정하는 최소 손실 감소 값
    "scale_pos_weight": [1, 3, 5, 10]                    # 만약 데이터가 '불균형'할 때만 사용 (1번 클래스 비율이 적을 때)
}

In [17]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print("탐색 범위 설정 완료")

탐색 범위 설정 완료


In [18]:
random_search = RandomizedSearchCV(
    estimator=xgb_for_random_search,
    param_distributions=param_distributions,
    n_iter=20,          #무작위 조합 20개
    scoring="roc_auc",  #ROC-AUC 기준으로 좋은 조합 찾기
    cv=cv,              #위에서 만든 3등분 교차검증
    random_state=42,
    n_jobs=-1           #CPU 최대한 활용
)

# train 데이터 안에서 탐색을 실행 (test는 사용하지 않음)
random_search.fit(X_train, y_train)

print("RandomizedSearchCV 탐색 완료")

RandomizedSearchCV 탐색 완료


In [19]:
#탐색 결과 확인
print("Best ROC-AUC:", random_search.best_score_)
print("Best Params:", random_search.best_params_)
print("Best Estimator:", random_search.best_estimator_)

Best ROC-AUC: 0.9918447434375413
Best Params: {'subsample': 0.6, 'scale_pos_weight': 10, 'reg_lambda': 1.0, 'reg_alpha': 0.01, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 0.6}
Best Estimator: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
           

In [20]:
# 탐색으로 찾은 가장 좋은 모델 꺼내기
xgb_random_search_model = random_search.best_estimator_

# test 데이터로 예측
xgb_random_search_pred = xgb_random_search_model.predict(X_test)
xgb_random_search_proba = xgb_random_search_model.predict_proba(X_test)[:, 1]

#공통 평가 함수로 정리
xgb_random_search_result = evaluate_classification_model(
    model_name="XGBoost RandomizedSearchCV",
    y_true=y_test,
    y_pred=xgb_random_search_pred,
    y_proba=xgb_random_search_proba
)

xgb_random_search_result

{'model_name': 'XGBoost RandomizedSearchCV',
 'accuracy': 0.9824561403508771,
 'precision_malignant': 0.9761904761904762,
 'recall_malignant': 0.9761904761904762,
 'f1_malignant': 0.9761904761904762,
 'roc_auc': 0.9970238095238095,
 'TN': np.int64(71),
 'FP': np.int64(1),
 'FN': np.int64(1),
 'TP': np.int64(41)}

In [25]:
def objective(trial):
    """
    Optuna가 XGBoost 하이퍼파라미터를 실험할 때 사용할 함수입니다.
    trial은 한 번의 실험을 의미합니다.
    Optuna는 trial마다 서로 다른 하이퍼파라미터 값을 넣어보고,
    그 결과 점수가 좋은지 확인합니다.
    """
    # trial.suggest_* : Optuna가 정해진 범위 안에서 값을 하나 골라줍니다.
    params = {
        # 트리 개수: 50~300 사이 정수 중에서 선택
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        #트리 깊이: 2~6 사이 정수
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        #학습 속도: 0.01~0.2, log=True는 작은 값 쪽을 더 촘촘히 탐색
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        #행 샘플링 비율: 0.7~1.0
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        # feature 샘플링 비율: 0.7~1.0
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        # L1 규제: 0.0~1.0
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        # L2 규제: 0.5~3.0
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
        # 재현성 고정
        "random_state": 42,
        "eval_metric": "logloss"
    }

    # 위에서 고른 설정값으로 XGBoost 모델을 만듭니다.
    model = XGBClassifier(**params)

    # train 데이터 안에서 3등분 교차검증
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    # 교차검증으로 ROC-AUC 점수들을 계산합니다. (test는 사용 안 함)
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )
    return scores.mean()

print("objective 함수 준비 완료")

objective 함수 준비 완료


In [26]:
# 로그가 너무 많이 출력되면 집중이 어려우므로 경고 수준만 보이도록 낮추기
optuna.logging.set_verbosity(optuna.logging.WARNING)

sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# objective 함수를 20번 실험합니다.
study.optimize(objective, n_trials=20)

print("Optuna 튜닝 완료")

Optuna 튜닝 완료


In [27]:
# 가장 좋았던 점수와 그 점수를 만든 설정값
print("Best ROC-AUC:", study.best_value)
print("Best Params:", study.best_params)

# 각 trial의 기록을 표로 확인 (상위 일부만)
study.trials_dataframe().head()

Best ROC-AUC: 0.9911082530888625
Best Params: {'n_estimators': 294, 'max_depth': 3, 'learning_rate': 0.06729392009628148, 'subsample': 0.77457520756364, 'colsample_bytree': 0.909182130134382, 'reg_alpha': 0.07977803963614018, 'reg_lambda': 2.659229016644547}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,state
0,0,0.990299,2026-06-18 20:08:53.300389,2026-06-18 20:08:53.451388,0 days 00:00:00.150999,0.746806,0.089608,6,144,0.155995,0.645209,0.879598,COMPLETE
1,1,0.990432,2026-06-18 20:08:53.451388,2026-06-18 20:08:53.656388,0 days 00:00:00.205000,0.990973,0.083411,5,267,0.832443,1.030848,0.706175,COMPLETE
2,2,0.986328,2026-06-18 20:08:53.656388,2026-06-18 20:08:53.739390,0 days 00:00:00.083002,0.829584,0.024879,2,95,0.291229,2.029632,0.857427,COMPLETE
3,3,0.987573,2026-06-18 20:08:53.739390,2026-06-18 20:08:53.836393,0 days 00:00:00.097003,0.935553,0.029967,3,85,0.199674,1.785586,0.836821,COMPLETE
4,4,0.990421,2026-06-18 20:08:53.836393,2026-06-18 20:08:53.976391,0 days 00:00:00.139998,0.719515,0.061721,2,198,0.948886,2.914080,0.751157,COMPLETE


In [28]:
xgb_optuna_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss"
)

xgb_optuna_model.fit(X_train, y_train)

xgb_optuna_pred = xgb_optuna_model.predict(X_test)
xgb_optuna_proba = xgb_optuna_model.predict_proba(X_test)[:, 1]

xgb_optuna_result = evaluate_classification_model(
    model_name="XGBoost Optuna",
    y_true=y_test,
    y_pred=xgb_optuna_pred,
    y_proba=xgb_optuna_proba
)

xgb_optuna_result

{'model_name': 'XGBoost Optuna',
 'accuracy': 0.9736842105263158,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9629629629629629,
 'roc_auc': 0.9923941798941799,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(3),
 'TP': np.int64(39)}

In [29]:
xgb_tuning_results_df = pd.DataFrame([
    xgb_base_result,
    xgb_random_search_result,
    xgb_optuna_result
])

xgb_tuning_results_df

,model_name,accuracy,precision_malignant,recall_malignant,f1_malignant,roc_auc,TN,FP,FN,TP
0,XGBoost Base,0.964912,1.00000,0.904762,0.950000,0.996693,72,0,4,38
1,XGBoost RandomizedSearchCV,0.982456,0.97619,0.976190,0.976190,0.997024,71,1,1,41
2,XGBoost Optuna,0.973684,1.00000,0.928571,0.962963,0.992394,72,0,3,39


In [30]:
logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_proba = logistic_model.predict_proba(X_test)[:, 1]

logistic_result = evaluate_classification_model(
    model_name="Logistic Regression",
    y_true=y_test,
    y_pred=logistic_pred,
    y_proba=logistic_proba
)

logistic_result

{'model_name': 'Logistic Regression',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 0.975,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9512195121951219,
 'roc_auc': 0.996031746031746,
 'TN': np.int64(71),
 'FP': np.int64(1),
 'FN': np.int64(3),
 'TP': np.int64(39)}

In [31]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_result = evaluate_classification_model(
    model_name="RandomForest",
    y_true=y_test,
    y_pred=rf_pred,
    y_proba=rf_proba
)

rf_result

{'model_name': 'RandomForest',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.994212962962963,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [33]:
# LightGBM 모델 (verbose=-1로 학습 로그를 끔)
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]

lgbm_result = evaluate_classification_model(
    model_name="LightGBM",
    y_true=y_test,
    y_pred=lgbm_pred,
    y_proba=lgbm_proba
)

lgbm_result

{'model_name': 'LightGBM',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9947089947089948,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}